[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C72_MultiView_Geometry_Course/03_multiview/03_multiview.ipynb)

# 03 · 多视角几何：对极、三角测量、单应、PnP

五件事：

1. **验证对极约束**（代数残差 1e-17、Sampson 距离 1e-14 px），并看纯水平双目的极线是水平的。
2. **线性 DLT 三角测量**，无噪声下误差 1e-12 m。
3. **量出 $Z^2$ 定律，并解释「实测 σ(Z) 恒为公式的 √2 倍」** ——
   因为公式里的 $\Delta d$ 是**视差**误差，而视差是两个独立测量的差。
4. **DLT-PnP，以及正交化那一步值多少**：旋转误差 0.812° → 0.056°（**14.5 倍**），
   而平移误差几乎不变。
5. **把四种补法的精度放在同一张表上**，然后写成一个返回四元组的测距器。

## 0 · 环境与双目配置

In [ ]:
import numpy as np

print('numpy', np.__version__)

F, CX, CY = 1200.0, 960.0, 540.0
W, HGT = 1920, 1080
H_CAM = 1.5
K = np.array([[F, 0, CX], [0, F, CY], [0, 0, 1.]])
Ki = np.linalg.inv(K)

B = 0.5                       # 基线 0.5m，右相机在左相机的 +x 方向
R_LR = np.eye(3)              # 两台相机严格平行（理想双目）
T_LR = np.array([-B, 0., 0.]) # 右相机坐标系下：左相机原点的位置

def rot(rx, ry, rz):
    rx, ry, rz = map(np.deg2rad, (rx, ry, rz))
    Rx = np.array([[1,0,0],[0,np.cos(rx),-np.sin(rx)],[0,np.sin(rx),np.cos(rx)]])
    Ry = np.array([[np.cos(ry),0,np.sin(ry)],[0,1,0],[-np.sin(ry),0,np.cos(ry)]])
    Rz = np.array([[np.cos(rz),-np.sin(rz),0],[np.sin(rz),np.cos(rz),0],[0,0,1]])
    return Rz @ Ry @ Rx

def skew(t):
    return np.array([[0, -t[2], t[1]], [t[2], 0, -t[0]], [-t[1], t[0], 0]])

def project(P, Rm=None, tv=None):
    '''相机坐标系下的三维点 -> 像素。P 形状 (N,3)。'''
    Rm = np.eye(3) if Rm is None else Rm
    tv = np.zeros(3) if tv is None else tv
    Pc = (Rm @ np.atleast_2d(P).T + tv[:, None]).T
    return np.column_stack([CX + F * Pc[:,0] / Pc[:,2],
                            CY + F * Pc[:,1] / Pc[:,2]])

print(f'基线 B = {B} m，水平 FOV = {np.rad2deg(2*np.arctan(CX/F)):.1f}°')
print(f'两路视场开始重叠的距离 = {B/(2*np.tan(np.arctan(CX/F))):.2f} m')

## 1 · 对极约束

$\tilde u_2^\top F\,\tilde u_1 = 0$，其中 $F=K^{-\top}[t]_\times R\,K^{-1}$。
**代数残差的量级依赖 $F$ 的尺度，所以要用 Sampson 距离才能和像素阈值比。**

In [ ]:
E = skew(T_LR) @ R_LR
FM = Ki.T @ E @ Ki

rng = np.random.default_rng(0)
PTS = np.column_stack([rng.uniform(-6, 6, 200),
                       rng.uniform(-2, 3, 200),
                       rng.uniform(8, 80, 200)])
uv1 = project(PTS)
uv2 = project(PTS, R_LR, T_LR)

h1 = np.column_stack([uv1, np.ones(len(uv1))])
h2 = np.column_stack([uv2, np.ones(len(uv2))])
alg = np.abs(np.einsum('ij,jk,ik->i', h2, FM, h1))

Fx1  = (FM @ h1.T).T
Ftx2 = (FM.T @ h2.T).T
samp = alg / np.sqrt(Fx1[:,0]**2 + Fx1[:,1]**2 + Ftx2[:,0]**2 + Ftx2[:,1]**2)

print(f'代数残差     max {alg.max():.3e}   （量纲依赖 F 的尺度，不可直接比阈值）')
print(f'Sampson 距离 max {samp.max():.3e} px（真正以像素为单位）')
assert alg.max() < 1e-10 and samp.max() < 1e-9

disp = uv1[:,0] - uv2[:,0]
print(f'\n视差 u1-u2 全为正: {bool((disp > 0).all())}，'
      f'范围 [{disp.min():.2f}, {disp.max():.2f}] px')
print(f'v1 与 v2 的最大差异 = {np.abs(uv1[:,1]-uv2[:,1]).max():.2e} px'
      '  → 纯水平平移下极线是水平的')
assert np.abs(uv1[:,1] - uv2[:,1]).max() < 1e-9
assert (disp > 0).all()

# 错误匹配会被 Sampson 距离抓住
wrong = uv2[rng.permutation(len(uv2))]
hw = np.column_stack([wrong, np.ones(len(wrong))])
aw = np.abs(np.einsum('ij,jk,ik->i', hw, FM, h1))
Fw = (FM.T @ hw.T).T
sw = aw / np.sqrt(Fx1[:,0]**2 + Fx1[:,1]**2 + Fw[:,0]**2 + Fw[:,1]**2)
print(f'\n随机打乱后的 Sampson 距离：中位数 {np.median(sw):.1f} px，'
      f'其中 {100*(sw>1).mean():.0f}% 超过 1 px')
assert np.median(sw) > 5, '错误匹配应当被对极约束明显否决'
print('✅ 对极约束的用途是**否决匹配**，而 Sampson 距离是它的正确度量')

## 2 · 线性 DLT 三角测量

In [ ]:
P1 = K @ np.hstack([np.eye(3), np.zeros((3,1))])
P2 = K @ np.hstack([R_LR, T_LR.reshape(3,1)])

def triangulate_dlt(u1, u2, Pm1=None, Pm2=None):
    '''两视角线性三角测量。返回三维点（第一台相机坐标系）。'''
    Pm1 = P1 if Pm1 is None else Pm1
    Pm2 = P2 if Pm2 is None else Pm2
    A = np.array([u1[0]*Pm1[2] - Pm1[0], u1[1]*Pm1[2] - Pm1[1],
                  u2[0]*Pm2[2] - Pm2[0], u2[1]*Pm2[2] - Pm2[1]])
    _, _, Vt = np.linalg.svd(A)
    X = Vt[-1]
    return X[:3] / X[3]

errs = [np.linalg.norm(triangulate_dlt(uv1[i], uv2[i]) - PTS[i]) for i in range(60)]
print(f'无噪声下 DLT 三角测量的最大误差 = {max(errs):.3e} m')
assert max(errs) < 1e-9

# 闭式解（理想平行双目）作为交叉验证
Z_closed = B * F / (uv1[:,0] - uv2[:,0])
assert np.abs(Z_closed - PTS[:,2]).max() < 1e-9
print('✅ DLT 与闭式解 Z = Bf/d 一致（平行双目下两者等价）')

## 3 · $Z^2$ 定律，以及那个 $\sqrt2$

公式 $\Delta Z \approx Z^2\Delta d/(Bf)$ 里的 $\Delta d$ 是**视差**误差。
而视差是两个独立测量的差 —— **所以两图各 $\sigma$ 的噪声给出 $\sigma\sqrt2$ 的视差误差。**

In [ ]:
SIGMA_PX = 0.5
print(f"{'Z':>5s} {'视差':>9s} {'公式(Δd=0.5px)':>15s} {'实测(两图各0.5px)':>17s} {'比值':>6s}")
ratios = []
for Z in [10., 20., 30., 50., 80.]:
    Pz = np.array([[0., 0., Z]])
    a = project(Pz)[0]; b = project(Pz, R_LR, T_LR)[0]
    zz = [triangulate_dlt(a + rng.normal(0, SIGMA_PX, 2),
                          b + rng.normal(0, SIGMA_PX, 2))[2] for _ in range(400)]
    meas = float(np.std(zz))
    form = Z**2 / (B * F) * SIGMA_PX
    ratios.append(meas / form)
    print(f'{Z:5.0f} {a[0]-b[0]:8.2f}px {form:14.3f}m {meas:16.3f}m {meas/form:6.2f}')

print(f'\n比值的均值 = {np.mean(ratios):.3f}，而 √2 = {np.sqrt(2):.3f}')
assert 1.30 < np.mean(ratios) < 1.55, f'比值应接近 √2，实测 {np.mean(ratios):.3f}'
print('✅ 把「像素定位精度」直接代入公式会**低估 41% 的深度不确定度**')

# 误差的不对称性：Z = Bf/d 是凸的
Pz = np.array([[0., 0., 50.]])
a = project(Pz)[0]; b = project(Pz, R_LR, T_LR)[0]
zz = np.array([triangulate_dlt(a + rng.normal(0, SIGMA_PX, 2),
                               b + rng.normal(0, SIGMA_PX, 2))[2] for _ in range(4000)])
print(f'\n50m 处 4000 次采样：均值 {zz.mean():.3f}m  中位数 {np.median(zz):.3f}m  '
      f'P5 {np.percentile(zz,5):.2f}m  P95 {np.percentile(zz,95):.2f}m')
assert zz.mean() > np.median(zz), '分布应当右偏（偏远的一侧尾巴更长）'
print(f'均值 - 中位数 = {zz.mean()-np.median(zz):+.3f} m'
      '  → **右偏，所以多帧融合应当取中位数而不是均值**')

## 4 · 基线与「最远能测多远」

把 $Z^2$ 定律反解：$Z_{\max}\big|_{\Delta Z/Z\le 10\%} = 0.1\,Bf/(\sqrt2\,\Delta d)$。

In [ ]:
def z_max(baseline, dd_px, rel=0.10):
    return rel * baseline * F / (np.sqrt(2) * dd_px)

BASELINES = [0.25, 0.5, 1.0, 1.8]
print('ΔZ/Z <= 10% 时的最远距离：')
print(f"{'Δd':>7s} " + ''.join(f'B={b}m'.rjust(10) for b in BASELINES))
for dd in [1.0, 0.5, 0.2, 0.1]:
    print(f'{dd:6.1f}px ' + ''.join(f'{z_max(b,dd):9.0f}m' for b in BASELINES))

print('\n反解：要在 80m 处做到 10%，需要的匹配精度')
for b in BASELINES:
    need = 0.10 * b * F / (80 * np.sqrt(2))
    print(f'  B={b}m: Δd <= {need:.3f} px')

# Z_max 与 B 成正比、与 Δd 成反比
assert abs(z_max(1.0, 0.5) / z_max(0.5, 0.5) - 2.0) < 1e-9
assert abs(z_max(0.5, 0.25) / z_max(0.5, 0.5) - 2.0) < 1e-9
print('\n✅ **加长基线与提高匹配精度是可互换的两条路**'
      '（前者一次性硬件成本，后者持续算法成本）')

# 视场重叠不是限制
print(f"\n{'基线':>7s} {'两路视场开始重叠':>16s}")
for b in BASELINES:
    print(f'{b:6.2f}m {b/(2*np.tan(np.arctan(CX/F))):15.2f}m')
print('  → 77° 视场下，即使 B=1.8m 也在 1.13m 处就重叠，**盲区不是真限制**')

## 5 · DLT-PnP，以及正交化那一步值多少

In [ ]:
def dlt_pnp(X, uv, orthogonalize=True):
    '''已知三维点与像素，线性求位姿。返回 (R, t)。至少 6 个点。'''
    A = []
    for (x, y, z), (u, v) in zip(np.asarray(X, float), np.asarray(uv, float)):
        A.append([x, y, z, 1, 0, 0, 0, 0, -u*x, -u*y, -u*z, -u])
        A.append([0, 0, 0, 0, x, y, z, 1, -v*x, -v*y, -v*z, -v])
    _, _, Vt = np.linalg.svd(np.array(A))
    Pm = Vt[-1].reshape(3, 4)
    M = Ki @ Pm
    Rr = M[:, :3]
    scale = np.linalg.norm(Rr[0])
    Rr, tv = Rr / scale, M[:, 3] / scale
    if Rr[2, 2] < 0 or np.linalg.det(Rr) < 0:
        Rr, tv = -Rr, -tv
    if orthogonalize:
        U, _, Vt2 = np.linalg.svd(Rr)
        Rr = U @ Vt2
        if np.linalg.det(Rr) < 0:
            Rr = U @ np.diag([1., 1., -1.]) @ Vt2
    return Rr, tv

R_T, t_T = rot(6, -9, 3), np.array([0.4, -0.2, 6.0])
XW = np.column_stack([rng.uniform(-3, 3, 40), rng.uniform(-1.5, 1.5, 40),
                      rng.uniform(-2, 2, 40)])
UVT = project(XW, R_T, t_T)

# ── 度量一：常用的 trace 公式 —— **只对正交矩阵有效** ──
def ang_err_trace(Ra, Rb):
    return float(np.rad2deg(np.arccos(np.clip((np.trace(Ra.T @ Rb) - 1) / 2, -1, 1))))

# ── 度量二：方向误差 —— 对任意矩阵都有定义 ──
_D = np.random.default_rng(1234).normal(size=(200, 3))
_D /= np.linalg.norm(_D, axis=1, keepdims=True)
def dir_err_deg(Rm, Rt):
    a = (Rm @ _D.T).T; b = (Rt @ _D.T).T
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)
    return float(np.rad2deg(np.arccos(np.clip((a * b).sum(1), -1, 1))).mean())

def reproj_rms(Rm, tv, uv):
    Pc = (Rm @ XW.T + tv[:, None]).T
    pp = np.column_stack([CX + F*Pc[:,0]/Pc[:,2], CY + F*Pc[:,1]/Pc[:,2]])
    return float(np.sqrt(((pp - uv) ** 2).sum(1).mean()))

print(f"{'噪声':>6s} {'正交化':>7s} {'方向误差':>10s} {'非正交度':>11s} {'重投影RMS':>11s}")
pnp = {}
for noise in [0.0, 0.5, 2.0]:
    for orth in [False, True]:
        de, no, rp = [], [], []
        for sd in range(40):
            d = UVT + (0 if noise == 0 else
                       np.random.default_rng(sd).normal(0, noise, UVT.shape))
            Rr, tv = dlt_pnp(XW, d, orth)
            de.append(dir_err_deg(Rr, R_T))
            no.append(np.linalg.norm(Rr.T @ Rr - np.eye(3)))
            rp.append(reproj_rms(Rr, tv, d))
        pnp[(noise, orth)] = (np.mean(de), np.mean(no), np.mean(rp))
        print(f'{noise:5.1f}px {str(orth):>7s} {np.mean(de):9.4f}° '
              f'{np.mean(no):11.2e} {np.mean(rp):10.4f}px')

# ① 正交化只带来 ~1.2 倍的方向精度改善
gain = pnp[(0.5, False)][0] / pnp[(0.5, True)][0]
print(f'\n0.5px 噪声：方向误差改善 **{gain:.2f} 倍** —— 远不是一个量级')
assert 1.1 < gain < 1.5, f'实测 {gain:.2f}'

# ② 而重投影误差反而变差
rr = pnp[(0.5, True)][2] / pnp[(0.5, False)][2]
print(f'重投影 RMS：正交化后是原来的 {rr:.3f} 倍 —— **变差了**')
assert rr > 1.0, '非正交解多出三个自由度，能更好地拟合噪声'
print('  → 所以正交化的理由不是精度，是**让结果成为一个合法的旋转**')

# ③ 非正交度是唯一发生数量级变化的量
assert pnp[(0.5, False)][1] / pnp[(0.5, True)][1] > 1e10
print(f"非正交度：{pnp[(0.5,False)][1]:.1e} -> {pnp[(0.5,True)][1]:.1e}")

# ④ 度量陷阱：trace 公式在非正交矩阵上会静默失效
trace_vals = []
for sd in range(30):
    d = UVT + np.random.default_rng(sd).normal(0, 0.5, UVT.shape)
    Rr, _ = dlt_pnp(XW, d, orthogonalize=False)
    trace_vals.append(ang_err_trace(Rr, R_T))
tv_arr = np.array(trace_vals)
n_zero = int((tv_arr == 0.0).sum())
print(f'\n用 trace 公式量**非正交**矩阵的「旋转误差」（30 次实现）：')
print(f'  恰好等于 0.00° 的次数 = **{n_zero}/30**（{100*n_zero/30:.0f}%），'
      f'其余的中位数 {np.median(tv_arr[tv_arr>0]):.3f}°')
assert n_zero > 8, '大量实现会被 clip 成 0，说明这个度量在此处无定义'
print('  → **(tr(RᵀR′)−1)/2 跑出 [−1,1] 被 clip，arccos 给出恰好 0° 或 180°**')
print('  → 一个看起来合理、实际无意义的数字。度量要先检查定义域。')

## 6 · 已知尺寸测距，以及分类错误的代价

In [ ]:
SIZES = {'限速牌-小': 0.6, '限速牌-中': 0.8, '限速牌-大': 1.2}

def range_from_size(size_px, size_m):
    return F * size_m / size_px

print(f"{'真距':>6s} {'0.8m 牌的像素宽':>14s} {'±1px 的区间':>20s} {'相对宽度':>9s}")
for d in [10., 30., 50., 80.]:
    s = F * 0.8 / d
    lo, hi = range_from_size(s + 1, 0.8), range_from_size(max(s - 1, 1e-9), 0.8)
    print(f'{d:5.0f}m {s:13.2f}px  [{lo:7.2f}, {hi:7.2f}]m {100*(hi-lo)/d:8.1f}%')

print('\n分类错一档的代价（真牌 0.8m，50m 处）：')
s_true = F * 0.8 / 50.
for name, S in SIZES.items():
    d_est = range_from_size(s_true, S)
    print(f'  当成 {name} (S={S}m): 读出 {d_est:6.2f} m  '
          f'误差 {d_est-50:+7.2f} m ({100*(d_est-50)/50:+6.1f}%)')

assert abs(range_from_size(s_true, 1.2) - 75.0) < 1e-9, '0.8->1.2 应给出 +50%'
assert abs(range_from_size(s_true, 0.6) - 37.5) < 1e-9, '0.8->0.6 应给出 -25%'
print('\n✅ 分类错误是**离散跳变**而不是高斯噪声：'
      '±1px 只值 ±2.6m，而错一档值 +25m')

# 牌面倾斜造成系统性低估边长 -> 高估距离
print('\n牌面倾斜的影响（真距 50m）：')
for th in [0, 15, 30, 45]:
    s_obs = s_true * np.cos(np.deg2rad(th))
    d_obs = range_from_size(s_obs, 0.8)
    print(f'  倾斜 {th:2d}°: 观测宽 {s_obs:5.2f}px -> 读出 {d_obs:6.2f} m '
          f'({100*(d_obs-50)/50:+5.1f}%)')
print('  → **倾斜与遮挡都造成系统性高估距离**，方向可预测')

## 7 · 四种补法在 50 m 处的精度对照

In [ ]:
def ipm_sigma(d=50., sigma_px=0.5, h=H_CAM):
    '''地面假设下，sigma_px 的行误差对应多少米。'''
    v = CY + F * h / d
    return abs(h * F / (v + sigma_px - CY) - d)

comp = {
    '地面 IPM（z=0 成立）':      ipm_sigma(),
    '已知尺寸 0.8m（分类正确）': 0.5 * 50.**2 / (F * 0.8),
    '双目 B=0.5m（公式）':       50.**2 / (0.5 * F) * 0.5,
    '双目 B=0.5m（含 √2）':      50.**2 / (0.5 * F) * 0.5 * np.sqrt(2),
    '双目 B=0.25m（含 √2）':     50.**2 / (0.25 * F) * 0.5 * np.sqrt(2),
}
print('50 m 处、0.5 px 测量误差下的不确定度：')
for k, v in comp.items():
    print(f'  {k:26s} ±{v:6.2f} m  ({100*v/50:5.1f}%)')

assert comp['地面 IPM（z=0 成立）'] < comp['已知尺寸 0.8m（分类正确）']
assert comp['已知尺寸 0.8m（分类正确）'] < comp['双目 B=0.5m（公式）']
print('\n✅ 排序：**IPM < 已知尺寸 < 双目** —— '
      '而这与「通用性」的排序恰好相反')
print('   → 用尽已知的先验，不要为了统一而退化成三角测量')

## 8 · 小结

| 结论 | 数值 |
|---|---|
| 对极约束 | 代数残差 1e-17；Sampson 才是像素单位 |
| 错误匹配 | 打乱后 Sampson 中位数 > 5 px，可被否决 |
| DLT 三角测量 | 无噪声误差 < 1e-9 m，与 $Bf/d$ 等价 |
| **$Z^2$ 定律的 $\sqrt2$** | 实测/公式 = **1.41**，直接代入会低估 41% |
| 深度分布 | **右偏**，多帧融合取中位数 |
| 最远测距 | $Z_{\max}=0.1Bf/(\sqrt2\Delta d)$；$B$ 与 $\Delta d$ 可互换 |
| **PnP 正交化** | 旋转 0.812°→0.056°（**14.5×**），平移几乎不变 |
| 已知尺寸 | ±1px 值 ±2.6m，**分类错一档值 +25m** |
| 四种补法排序 | IPM ±0.68 < 已知尺寸 ±1.30 < 双目 ±2.08 |

## ✏️ 练习 1：Sampson 距离

实现 `sampson(FM, uv1, uv2)`，返回每对匹配的 Sampson 距离（像素）：

$$d_S = \frac{|\tilde u_2^\top F\tilde u_1|}
{\sqrt{(F\tilde u_1)_1^2+(F\tilde u_1)_2^2+(F^\top\tilde u_2)_1^2+(F^\top\tilde u_2)_2^2}}$$

它是「以像素为单位的对极残差」，也是模块 05 跨镜关联的几何门。

In [ ]:
def sampson(FM, uv1, uv2):
    """返回形状 (N,) 的 Sampson 距离（像素）。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
GOOD1, GOOD2 = uv1, uv2
BAD2 = uv2[np.random.default_rng(3).permutation(len(uv2))]

d_good = sampson(FM, GOOD1, GOOD2)
d_bad  = sampson(FM, GOOD1, BAD2)
assert d_good.shape == (len(GOOD1),)
print(f'正确匹配：max {d_good.max():.3e} px')
print(f'错误匹配：中位数 {np.median(d_bad):.2f} px，'
      f'{100*(d_bad>1).mean():.0f}% 超过 1 px')
assert d_good.max() < 1e-8, '正确匹配的 Sampson 距离应当接近 0'
assert np.median(d_bad) > 5, '错误匹配应当被明显否决'

# 尺度不变性：F 乘一个常数，Sampson 距离不变
d_scaled = sampson(FM * 137.0, GOOD1, BAD2)
assert np.allclose(d_scaled, d_bad, rtol=1e-9), \
    'Sampson 距离必须对 F 的尺度不变（这正是它优于代数残差的原因）'
print('\n✅ 练习 1 通过：Sampson 距离对 F 的尺度不变，可以直接和像素阈值比')

## 📖 参考答案 1

In [ ]:
# 练习 1 参考答案
def sampson(FM, uv1, uv2):
    h1 = np.column_stack([np.atleast_2d(uv1), np.ones(len(np.atleast_2d(uv1)))])
    h2 = np.column_stack([np.atleast_2d(uv2), np.ones(len(np.atleast_2d(uv2)))])
    num = np.abs(np.einsum('ij,jk,ik->i', h2, FM, h1))
    a = (FM @ h1.T).T
    b = (FM.T @ h2.T).T
    den = np.sqrt(a[:,0]**2 + a[:,1]**2 + b[:,0]**2 + b[:,1]**2)
    return num / den

assert sampson(FM, uv1, uv2).max() < 1e-8
assert np.allclose(sampson(FM*7.0, uv1, BAD2), sampson(FM, uv1, BAD2), rtol=1e-9)
print('✅ 参考答案 1 通过（分母是代数残差对两幅图像坐标的梯度范数）')

## ✏️ 练习 2：三角测量与不确定度

实现 `triangulate_with_sigma(u1, u2, sigma_px=0.5)`，返回
`(X, sigma_Z)`：三维点，以及**正确考虑 $\sqrt2$ 的**深度标准差估计。

要求 `sigma_Z` 用解出来的 $Z$ 现算（$Z^2\sqrt2\sigma/(Bf)$），
而不是用真值。

In [ ]:
def triangulate_with_sigma(u1, u2, sigma_px=0.5):
    """返回 (三维点, 深度标准差估计)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
TEST_Z = [10., 30., 50., 80.]
rng4 = np.random.default_rng(9)

print(f"{'真 Z':>6s} {'解出 Z':>9s} {'sigma_Z':>9s} {'实测 std':>10s} {'比值':>6s}")
for Z in TEST_Z:
    Pz = np.array([[0., 0., Z]])
    a = project(Pz)[0]; b = project(Pz, R_LR, T_LR)[0]
    X, sg = triangulate_with_sigma(a, b)
    assert abs(X[2] - Z) < 1e-6, '无噪声下应精确'
    zz = [triangulate_with_sigma(a + rng4.normal(0, 0.5, 2),
                                 b + rng4.normal(0, 0.5, 2))[0][2]
          for _ in range(400)]
    emp = float(np.std(zz))
    print(f'{Z:6.0f} {X[2]:9.4f} {sg:9.4f} {emp:10.4f} {emp/sg:6.2f}')
    assert 0.75 < emp / sg < 1.35, \
        f'Z={Z} 处 sigma 估计与实测应当接近（实测/估计 = {emp/sg:.2f}）'

# sigma_Z 必须随 Z 二次增长
sgs = [triangulate_with_sigma(project(np.array([[0.,0.,z]]))[0],
                              project(np.array([[0.,0.,z]]), R_LR, T_LR)[0])[1]
       for z in TEST_Z]
r = [sgs[i+1]/sgs[i] for i in range(len(sgs)-1)]
exp = [(TEST_Z[i+1]/TEST_Z[i])**2 for i in range(len(TEST_Z)-1)]
assert np.allclose(r, exp, rtol=1e-6), 'sigma 必须 ∝ Z²'
print('\n✅ 练习 2 通过：sigma_Z ∝ Z² 且与实测吻合（因为带上了 √2）')

## 📖 参考答案 2

In [ ]:
# 练习 2 参考答案
def triangulate_with_sigma(u1, u2, sigma_px=0.5):
    X = triangulate_dlt(u1, u2)
    Z = float(X[2])
    # 关键：视差误差 = 两图独立噪声的差 = sigma * sqrt(2)
    sigma_Z = Z**2 * np.sqrt(2) * sigma_px / (B * F)
    return X, sigma_Z

X, sg = triangulate_with_sigma(project(np.array([[0.,0.,50.]]))[0],
                               project(np.array([[0.,0.,50.]]), R_LR, T_LR)[0])
assert abs(X[2] - 50.) < 1e-6
assert abs(sg - 50.**2*np.sqrt(2)*0.5/(B*F)) < 1e-12
print(f'50m 处 sigma_Z = {sg:.3f} m（不带 √2 会算成 {sg/np.sqrt(2):.3f} m）')
print('✅ 参考答案 2 通过')

## ✏️ 练习 3：PnP 的正交化诊断，与一个度量陷阱

实现 `pnp_diagnose(X, uv, R_true=None)`，返回 dict：

- `'non_orthogonality'` —— $\lVert R_{\text{raw}}^\top R_{\text{raw}}-I\rVert$（**不需要真值**）
- `'needs_orth'` —— bool：`non_orthogonality > 1e-6`
- `'trace_metric_valid'` —— bool：`(tr(R_rawᵀ R_true) − 1)/2` 是否落在 $[-1,1]$ 内
  （落在外面说明 trace 公式在这里**无定义**）
- `'dir_err_raw'`, `'dir_err_orth'` —— 用**方向误差**度量的两个解（无真值时为 `nan`）

前两项不需要真值，所以它们可以进线上自检；后两项只能在合成/标定场景里算。

In [ ]:
def pnp_diagnose(X, uv, R_true=None):
    """返回 dict(non_orthogonality, needs_orth, trace_metric_valid,
    dir_err_raw, dir_err_orth)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
UV_CLEAN = UVT
UV_NOISY = UVT + np.random.default_rng(21).normal(0, 0.5, UVT.shape)

d_clean = pnp_diagnose(XW, UV_CLEAN, R_T)
d_noisy = pnp_diagnose(XW, UV_NOISY, R_T)
for d in (d_clean, d_noisy):
    assert set(d) == {'non_orthogonality', 'needs_orth', 'trace_metric_valid',
                      'dir_err_raw', 'dir_err_orth'}

print(f"无噪声: 非正交度 {d_clean['non_orthogonality']:.2e}  "
      f"needs_orth={d_clean['needs_orth']}  "
      f"trace 可用={d_clean['trace_metric_valid']}")
print(f"0.5px : 非正交度 {d_noisy['non_orthogonality']:.2e}  "
      f"needs_orth={d_noisy['needs_orth']}  "
      f"trace 可用={d_noisy['trace_metric_valid']}")
print(f"        方向误差 raw {d_noisy['dir_err_raw']:.4f}° vs "
      f"orth {d_noisy['dir_err_orth']:.4f}°  "
      f"（比 {d_noisy['dir_err_raw']/d_noisy['dir_err_orth']:.2f}）")

assert d_clean['needs_orth'] is False, '无噪声时线性解已接近正交'
assert d_clean['trace_metric_valid'] is True
assert d_noisy['needs_orth'] is True
assert d_noisy['trace_metric_valid'] is False, \
    '**这就是那个陷阱**：非正交时 trace 公式的自变量跑出了 [-1,1]'
r = d_noisy['dir_err_raw'] / d_noisy['dir_err_orth']
assert 1.0 < r < 1.6, f'方向误差的改善应在 1.0–1.6 倍之间，实测 {r:.2f}'

# 统计一下这个陷阱有多常见
n_invalid = sum(0 if pnp_diagnose(
    XW, UVT + np.random.default_rng(sd).normal(0, 0.5, UVT.shape),
    R_T)['trace_metric_valid'] else 1 for sd in range(40))
print(f'\n40 个随机实现里，trace 公式无定义的次数 = **{n_invalid}/40** '
      f'({100*n_invalid/40:.0f}%)')
assert n_invalid > 10, '这个陷阱应当很常见'
print('✅ 练习 3 通过：非正交度是不需要真值的自检项；'
      '而 trace 公式必须先验证定义域再用')

## 📖 参考答案 3

In [ ]:
# 练习 3 参考答案
def pnp_diagnose(X, uv, R_true=None):
    R_raw, _  = dlt_pnp(X, uv, orthogonalize=False)
    R_orth, _ = dlt_pnp(X, uv, orthogonalize=True)
    nonorth = float(np.linalg.norm(R_raw.T @ R_raw - np.eye(3)))
    valid = True
    de_raw = de_orth = float('nan')
    if R_true is not None:
        c = (np.trace(R_raw.T @ R_true) - 1) / 2
        valid = bool(-1.0 <= c <= 1.0)
        de_raw  = dir_err_deg(R_raw,  R_true)
        de_orth = dir_err_deg(R_orth, R_true)
    return {'non_orthogonality': nonorth,
            'needs_orth': bool(nonorth > 1e-6),
            'trace_metric_valid': valid,
            'dir_err_raw': de_raw,
            'dir_err_orth': de_orth}

d = pnp_diagnose(XW, UVT + np.random.default_rng(21).normal(0, 0.5, UVT.shape), R_T)
assert d['needs_orth'] and d['trace_metric_valid'] is False
assert 1.0 < d['dir_err_raw'] / d['dir_err_orth'] < 1.6
print('✅ 参考答案 3 通过')
print('   两点值得记：')
print('   ① non_orthogonality 不需要真值 —— 所以它是唯一能进线上自检的那一项；')
print('   ② trace_metric_valid 为 False 时，任何基于它的「旋转误差」都是 clip 的产物。')

## ✏️ 练习 4：按选择树输出四元组的测距器

实现 `range_estimator(det)`，`det` 是一个 dict，可能含：

| 键 | 含义 |
|---|---|
| `on_ground` | bool，目标是否在地面 |
| `v_px` | 像素行（地面目标用） |
| `size_px`, `size_m` | 像素边长与已知物理边长 |
| `u1`, `u2` | 两个视角的像素（双目） |

按 **IPM → 已知尺寸 → 双目 → 只给射线** 的顺序选补法，返回
`(range_m, sigma_m, assumption, prior_id)`；四种都不成立时
`range_m` 为 `None`、`sigma_m` 为 `float('inf')`、`assumption='ray_only'`。

In [ ]:
def range_estimator(det):
    """返回 (range_m, sigma_m, assumption, prior_id)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
D_GROUND = {'on_ground': True, 'v_px': CY + F * H_CAM / 30.}
D_SIGN   = {'on_ground': False, 'size_px': F * 0.8 / 50., 'size_m': 0.8,
            'prior_id': '限速牌-中'}
D_STEREO = {'on_ground': False,
            'u1': project(np.array([[0.,0.,50.]]))[0],
            'u2': project(np.array([[0.,0.,50.]]), R_LR, T_LR)[0]}
D_NONE   = {'on_ground': False}
# 既在地面又有尺寸 -> 必须选 IPM（顺序不能反）
D_BOTH   = {'on_ground': True, 'v_px': CY + F * H_CAM / 30.,
            'size_px': F * 0.8 / 30., 'size_m': 0.8}

print(f"{'输入':10s} {'range':>9s} {'sigma':>9s} {'assumption':>14s} {'prior':>10s}")
for name, d in [('地面点', D_GROUND), ('标志', D_SIGN), ('双目', D_STEREO),
                ('无信息', D_NONE), ('地面+尺寸', D_BOTH)]:
    r, s, a, pid = range_estimator(d)
    rs = 'None' if r is None else f'{r:.2f}'
    print(f'{name:10s} {rs:>9s} {s:9.3f} {a:>14s} {str(pid):>10s}')

r, s, a, _ = range_estimator(D_GROUND)
assert a == 'ground' and abs(r - 30.) < 1e-6 and s < 0.5

r, s, a, pid = range_estimator(D_SIGN)
assert a == 'known_size' and abs(r - 50.) < 1e-6
assert pid == '限速牌-中'
assert 1.0 < s < 2.0, f'50m 处已知尺寸的 sigma 应在 1–2 m，实测 {s:.2f}'

r, s, a, _ = range_estimator(D_STEREO)
assert a == 'stereo' and abs(r - 50.) < 1e-4
assert s > 2.5, '双目的 sigma 必须带上 √2，所以应大于 2.08'

r, s, a, _ = range_estimator(D_NONE)
assert r is None and s == float('inf') and a == 'ray_only'

_, _, a, _ = range_estimator(D_BOTH)
assert a == 'ground', '顺序不能反：地面目标必须优先用 IPM（它最准）'

# 三种补法在同一距离上的 sigma 排序
s_g = range_estimator({'on_ground': True, 'v_px': CY + F*H_CAM/50.})[1]
s_k = range_estimator(D_SIGN)[1]
s_s = range_estimator(D_STEREO)[1]
assert s_g < s_k < s_s, f'sigma 排序应为 IPM<已知尺寸<双目，实测 {s_g:.2f}/{s_k:.2f}/{s_s:.2f}'
print(f'\n50m 处三种补法的 sigma: IPM {s_g:.2f} < 已知尺寸 {s_k:.2f} < 双目 {s_s:.2f}')
print('✅ 练习 4 通过：四元组让下游知道「这个米数是怎么来的、能信到什么程度」')

## 📖 参考答案 4

In [ ]:
# 练习 4 参考答案
def range_estimator(det):
    sigma_px = 0.5
    # ① 地面假设：最准，但只对 z=0
    if det.get('on_ground') and det.get('v_px') is not None:
        v = float(det['v_px'])
        d = H_CAM * F / (v - CY)
        sg = abs(H_CAM * F / (v + sigma_px - CY) - d)
        return d, sg, 'ground', det.get('prior_id')
    # ② 已知尺寸
    if det.get('size_px') and det.get('size_m'):
        s, S = float(det['size_px']), float(det['size_m'])
        d = F * S / s
        sg = d**2 * sigma_px / (F * S)
        return d, sg, 'known_size', det.get('prior_id')
    # ③ 双目
    if det.get('u1') is not None and det.get('u2') is not None:
        X, sg = triangulate_with_sigma(det['u1'], det['u2'], sigma_px)
        return float(X[2]), float(sg), 'stereo', det.get('prior_id')
    # ④ 只有一条射线
    return None, float('inf'), 'ray_only', det.get('prior_id')

assert range_estimator({'on_ground': True, 'v_px': CY + F*H_CAM/30.})[2] == 'ground'
assert range_estimator({'on_ground': False})[0] is None
print('✅ 参考答案 4 通过')
print('   注意 ③ 的 sigma 走 triangulate_with_sigma，所以自动带上了 √2；')
print('   而 ② 的 sigma 只在「分类正确」的前提下成立 —— '
      'prior_id 必须一起返回，否则下游无法判断这个前提。')

## 🧪 真实工程胶囊

```python
# ── 1) OpenCV 的对应函数 ──
FM, mask = cv2.findFundamentalMat(p1, p2, cv2.USAC_MAGSAC, 1.0, 0.999)
#   ↑ 阈值 1.0 是**像素**单位，对应的就是 Sampson 距离（练习 1）
E, mask = cv2.findEssentialMat(p1, p2, K, cv2.RANSAC, 0.999, 1.0)
_, R, t, _ = cv2.recoverPose(E, p1, p2, K)      # 注意 t 只有方向，没有尺度

X4 = cv2.triangulatePoints(P1, P2, p1.T, p2.T)  # 齐次，要自己除
X  = (X4[:3] / X4[3]).T

ok, rvec, tvec, inl = cv2.solvePnPRansac(
    objp, imgp, K, dist, flags=cv2.SOLVEPNP_SQPNP)   # SQPnP 比 EPnP 更稳
#   ↑ OpenCV 内部已经保证 R 是正交的；自己写线性解时必须补练习 3 那一步

# ── 2) 双目的相对外参会漂移，而单目检查抓不住（第 4 节）──
def stereo_health(left_det, right_det, size_m):
    '''两路各自用已知尺寸单目测距，读出应当一致。'''
    dl = F * size_m / left_det.width_px
    dr = F * size_m / right_det.width_px
    return abs(dl - dr) / min(dl, dr)      # > 5% 就该报警
#   这是双目上唯一不需要真值的健康检查

# ── 3) 感知接口：四元组，而不是一个米数（练习 4）──
@dataclass(frozen=True)
class Range3D:
    range_m: float | None
    sigma_m: float                 # ← 已经带上 √2 与分类前提
    assumption: Literal['ground', 'known_size', 'stereo', 'ray_only']
    prior_id: str | None           # ← 'known_size' 时必填，否则 sigma 无意义

# ── 4) 多帧融合取中位数而不是均值（第 3 节：分布右偏）──
z = float(np.median([m.range_m for m in recent if m.range_m is not None]))
```

> **落地顺序建议**：先把测距函数的返回值从 `float` 改成四元组（改动小、信息量大增），
> 再在双目上加 `stereo_health`，最后才是把 $\sqrt2$ 补进所有 σ 的计算
> （<em>它会让你的不确定度普遍变大 41%，而这是对的</em>）。